# Sobreajuste y Regularización en MNIST

En este notebook entrenamos 6 clasificadores sobre MNIST con la **misma arquitectura** pero distintas combinaciones de regularización y aumento de datos, para ver cómo cada técnica afecta el sobreajuste.

| # | Modelo | Dropout | Weight Decay | Dataset |
|---|--------|---------|--------------|----------|
| M1 | Base | ✗ | ✗ | Original |
| M2 | Aumentado | ✗ | ✗ | Aumentado |
| M3 | Weight Decay | ✗ | ✓ | Original |
| M4 | Weight Decay + Aug | ✗ | ✓ | Aumentado |
| M5 | Dropout | ✓ | ✗ | Original |
| M6 | Dropout + WD + Aug | ✓ | ✓ | Aumentado |

In [ ]:
!pip install torch torchvision tqdm matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Usando: {device}')

## Datos

Cargamos MNIST en dos versiones:
- **Original**: normalización estándar
- **Aumentado**: rotaciones (±15°), traslaciones (±10%) y escalado (90–110%)

El conjunto de test **nunca** se aumenta.

In [ ]:
BATCH_SIZE = 64

normalize = transforms.Normalize((0.1307,), (0.3081,))

transform_original = transforms.Compose([
    transforms.ToTensor(),
    normalize,
])

transform_augmented = transforms.Compose([
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.10, 0.10), scale=(0.90, 1.10)),
    transforms.ToTensor(),
    normalize,
])

train_original = datasets.MNIST(root='data', train=True,  download=True, transform=transform_original)
train_augmented = datasets.MNIST(root='data', train=True,  download=True, transform=transform_augmented)
test_set       = datasets.MNIST(root='data', train=False, download=True, transform=transform_original)

loader_original  = DataLoader(train_original,  batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
loader_augmented = DataLoader(train_augmented, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
loader_test      = DataLoader(test_set,        batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Visualizar ejemplos originales vs aumentados
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle('Ejemplos: original (arriba) vs aumentado (abajo)', fontsize=13)
orig_imgs  = [train_original[i][0]  for i in range(8)]
aug_imgs   = [train_augmented[i][0] for i in range(8)]
for col in range(8):
    axes[0, col].imshow(orig_imgs[col].squeeze(), cmap='gray')
    axes[0, col].axis('off')
    axes[1, col].imshow(aug_imgs[col].squeeze(), cmap='gray')
    axes[1, col].axis('off')
plt.tight_layout()
plt.show()

## Arquitectura

Todos los modelos tienen **la misma cantidad de capas y neuronas**:

```
784 → 512 → 256 → 128 → 64 → 10
```

El parámetro `use_dropout` añade una capa `Dropout(0.5)` después de cada ReLU sin cambiar el número de neuronas.

In [ ]:
class MNISTClassifier(nn.Module):
    """Clasificador MLP para MNIST. Misma arquitectura para todos los experimentos."""

    def __init__(self, use_dropout: bool = False):
        super().__init__()
        p = 0.5 if use_dropout else 0.0

        layers = []
        sizes = [784, 512, 256, 128, 64]
        for in_f, out_f in zip(sizes, sizes[1:]):
            layers.append(nn.Linear(in_f, out_f))
            layers.append(nn.ReLU())
            if use_dropout:
                layers.append(nn.Dropout(p))
        layers.append(nn.Linear(64, 10))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

## Función de entrenamiento

In [ ]:
def train_model(model, train_loader, test_loader, epochs=20, weight_decay=0.0, label='modelo'):
    """Entrena el modelo y devuelve historial de pérdida y accuracy."""
    model = model.to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)

    train_losses, test_losses = [], []
    train_accs,   test_accs   = [], []

    pbar = tqdm(range(epochs), desc=label, leave=True)
    for epoch in pbar:
        # --- Entrenamiento ---
        model.train(True)
        running_loss = 0.0
        correct, total = 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = loss_fn(out, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            correct += (out.argmax(1) == labels).sum().item()
            total   += labels.size(0)
        train_losses.append(running_loss / len(train_loader))
        train_accs.append(correct / total)

        # --- Evaluación en test ---
        model.eval()
        test_loss = 0.0
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                out = model(imgs)
                test_loss += loss_fn(out, labels).item()
                correct   += (out.argmax(1) == labels).sum().item()
                total     += labels.size(0)
        test_losses.append(test_loss / len(test_loader))
        test_accs.append(correct / total)

        pbar.set_postfix({
            'train_loss': f'{train_losses[-1]:.4f}',
            'test_loss':  f'{test_losses[-1]:.4f}',
            'test_acc':   f'{test_accs[-1]:.3f}',
        })

    return {
        'label':        label,
        'train_losses': train_losses,
        'test_losses':  test_losses,
        'train_accs':   train_accs,
        'test_accs':    test_accs,
    }

## M1 — Base (sin regularización, datos originales)

In [ ]:
EPOCHS = 20

m1 = MNISTClassifier(use_dropout=False)
hist_m1 = train_model(m1, loader_original, loader_test, epochs=EPOCHS,
                      weight_decay=0.0, label='M1: Base')

## M2 — Aumento de datos (sin regularización, datos aumentados)

In [ ]:
m2 = MNISTClassifier(use_dropout=False)
hist_m2 = train_model(m2, loader_augmented, loader_test, epochs=EPOCHS,
                      weight_decay=0.0, label='M2: Aumentado')

## M3 — Weight Decay (datos originales)

In [ ]:
m3 = MNISTClassifier(use_dropout=False)
hist_m3 = train_model(m3, loader_original, loader_test, epochs=EPOCHS,
                      weight_decay=1e-4, label='M3: Weight Decay')

## M4 — Weight Decay + Aumento de datos

In [ ]:
m4 = MNISTClassifier(use_dropout=False)
hist_m4 = train_model(m4, loader_augmented, loader_test, epochs=EPOCHS,
                      weight_decay=1e-4, label='M4: WD + Aumentado')

## M5 — Dropout (datos originales)

In [ ]:
m5 = MNISTClassifier(use_dropout=True)
hist_m5 = train_model(m5, loader_original, loader_test, epochs=EPOCHS,
                      weight_decay=0.0, label='M5: Dropout')

## M6 — Dropout + Weight Decay + Aumento de datos

In [ ]:
m6 = MNISTClassifier(use_dropout=True)
hist_m6 = train_model(m6, loader_augmented, loader_test, epochs=EPOCHS,
                      weight_decay=1e-4, label='M6: Dropout + WD + Aumentado')

## Curvas de pérdida

Para cada modelo se muestra la pérdida en entrenamiento (azul) y en test (naranja). Una brecha grande indica sobreajuste.

In [ ]:
all_hists = [hist_m1, hist_m2, hist_m3, hist_m4, hist_m5, hist_m6]
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Curvas de pérdida por modelo', fontsize=15, fontweight='bold')

for ax, hist in zip(axes.flat, all_hists):
    ax.plot(epochs_range, hist['train_losses'], label='Entrenamiento', color='steelblue',  linewidth=2)
    ax.plot(epochs_range, hist['test_losses'],  label='Test',          color='darkorange', linewidth=2)
    ax.set_title(hist['label'], fontsize=12)
    ax.set_xlabel('Época')
    ax.set_ylabel('Pérdida (Cross-Entropy)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Rendimiento final

Comparamos la pérdida y el accuracy en la última época, tanto en entrenamiento como en test.

In [ ]:
# --- Tabla de resultados ---
print(f"{'Modelo':<30} {'Train Loss':>10} {'Test Loss':>10} {'Train Acc':>10} {'Test Acc':>10} {'Gap Acc':>10}")
print('-' * 80)
for h in all_hists:
    tr_loss = h['train_losses'][-1]
    te_loss = h['test_losses'][-1]
    tr_acc  = h['train_accs'][-1]
    te_acc  = h['test_accs'][-1]
    gap     = tr_acc - te_acc
    print(f"{h['label']:<30} {tr_loss:>10.4f} {te_loss:>10.4f} {tr_acc:>10.3f} {te_acc:>10.3f} {gap:>10.3f}")

In [ ]:
# --- Gráfico de barras: accuracy en entrenamiento y test ---
labels_short = ['M1\nBase', 'M2\nAug', 'M3\nWD', 'M4\nWD+Aug', 'M5\nDropout', 'M6\nAll']
train_accs_final = [h['train_accs'][-1] for h in all_hists]
test_accs_final  = [h['test_accs'][-1]  for h in all_hists]

x = np.arange(len(labels_short))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars_train = ax.bar(x - width/2, train_accs_final, width, label='Train Acc', color='steelblue',  alpha=0.85)
bars_test  = ax.bar(x + width/2, test_accs_final,  width, label='Test Acc',  color='darkorange', alpha=0.85)

# Anotaciones
for bar in bars_train:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
for bar in bars_test:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Accuracy')
ax.set_title('Accuracy final en entrenamiento y test por modelo', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels_short)
ax.set_ylim(0.9, 1.01)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Brecha entre train acc y test acc (sobreajuste) ---
gaps = [tr - te for tr, te in zip(train_accs_final, test_accs_final)]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#d62728' if g > 0.01 else '#2ca02c' for g in gaps]
bars = ax.bar(labels_short, gaps, color=colors, alpha=0.85)
for bar, g in zip(bars, gaps):
    ax.annotate(f'{g:.4f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=10)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Train Acc − Test Acc')
ax.set_title('Brecha de sobreajuste por modelo\n(rojo = sobreajuste notable, verde = generalización aceptable)',
             fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusiones

| Técnica | Efecto esperado |
|---------|----------------|
| **Sin regularización (M1)** | Mayor brecha train/test → sobreajuste |
| **Aumento de datos (M2)** | El modelo ve más variación → mejor generalización |
| **Weight Decay (M3)** | Penaliza pesos grandes → modelos más simples |
| **WD + Aug (M4)** | Combinación de ambas → regularización más fuerte |
| **Dropout (M5)** | Apaga neuronas aleatoriamente → evita co-adaptación |
| **Dropout + WD + Aug (M6)** | Regularización máxima → generalización óptima |

En MNIST el sobreajuste es limitado por la simplicidad del dataset, pero las curvas de pérdida y la brecha train−test ilustran claramente el efecto de cada técnica.